# 第157章: 埋め込みの応用と統合 (Embedding Applications & Synthesis)

## Capstone: RAG, Clustering, Bias Analysis & Multilingual Embeddings

---

### このノートブックの位置づけ

**Embeddingsシリーズ** の最終章（第8章）として、150-156で学んだ知識を **実践的な応用** に統合します。RAG（検索拡張生成）、クラスタリング、バイアス分析、多言語埋め込みなど、現代のNLPで不可欠なトピックを扱います。

### 学習目標

この章を終えると、以下ができるようになります：

- [ ] RAG（検索拡張生成）の仕組みを理解し、簡易パイプラインを構築できる
- [ ] 埋め込みを用いたクラスタリング分析を実施できる
- [ ] 埋め込みに含まれるバイアスを検出・可視化できる
- [ ] 多言語埋め込みの概念と応用を理解できる
- [ ] Embeddingsシリーズ全体の知識を統合し、実践に応用できる

### 🎯 前提知識

- ✅ Notebook 150-156（Embeddingsシリーズ全体）
- ✅ PyTorch基礎（Notebook 35-36）

⏱️ **推定学習時間**: 90-120分
📊 **難易度**: ★★★☆☆（中級）
🎓 **カテゴリ**: Embeddings / 応用

## 目次

1. [環境セットアップ](#1-環境セットアップ)
2. [シリーズの振り返り — 150-156の知識マップ](#2-シリーズの振り返り)
3. [RAG（検索拡張生成）の実装](#3-rag検索拡張生成の実装)
4. [埋め込みによるクラスタリング](#4-埋め込みによるクラスタリング)
5. [埋め込みのバイアス分析](#5-埋め込みのバイアス分析)
6. [多言語埋め込み](#6-多言語埋め込み)
7. [Capstoneプロジェクト — 統合演習](#7-capstoneプロジェクト)
8. [Embeddingsシリーズ総括](#8-embeddingsシリーズ総括)
9. [まとめ・よくある間違い・自己評価クイズ](#9-まとめ)

---

## 1. 環境セットアップ

In [ ]:
# ============================================================
# Section 1: 環境セットアップ
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import (
    adjusted_rand_score, normalized_mutual_info_score,
    silhouette_score, silhouette_samples
)
from sklearn.decomposition import PCA
import warnings

warnings.filterwarnings('ignore')

# ============================================================
# 日本語フォントの設定
# ============================================================
import matplotlib.font_manager as fm

def setup_japanese_font():
    """日本語フォントを設定する"""
    japanese_fonts = [
        'Hiragino Sans', 'Hiragino Maru Gothic Pro', 'AppleGothic',
        'Yu Gothic', 'MS Gothic',
        'Noto Sans CJK JP', 'IPAexGothic', 'TakaoPGothic',
    ]
    available_fonts = set(f.name for f in fm.fontManager.ttflist)
    for font in japanese_fonts:
        if font in available_fonts:
            plt.rcParams['font.family'] = font
            plt.rcParams['axes.unicode_minus'] = False
            return font
    return None

font_used = setup_japanese_font()

# ============================================================
# スタイルとシード
# ============================================================
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)
np.random.seed(42)

print("✅ 基本ライブラリのインポート完了")
if font_used:
    print(f"   日本語フォント: {font_used}")
print(f"   NumPy: {np.__version__}")

---

## 2. シリーズの振り返り — 150-156の知識マップ

### Embeddingsシリーズの全体像

```
150: 幾何学の基礎
 │  (距離・類似度・高次元の呪い)
 ▼
151: Word2Vec と静的埋め込み
 │  (Skip-gram実装・GloVe・FastText)
 ▼
152: 文脈付き埋め込み
 │  (BERT層別分析・多義語・Attention)
 ▼
153: 文・文書の埋め込み
 │  (Sentence-BERT・プーリング・意味検索)
 ▼
154: 多様体学習と可視化          155: ベクトル検索
 │  (PCA/t-SNE/UMAP)            │  (FAISS/IVF/HNSW)
 ▼                               ▼
156: 距離学習とファインチューニング
 │  (Triplet Loss・Hard Mining)
 ▼
157: 応用と統合 ← 今ここ！
    (RAG・クラスタリング・バイアス・多言語)
```

### 各章の核心概念

| # | 核心 | キーワード |
|---|------|--------|
| 150 | ベクトル空間の意味 | コサイン類似度, 高次元の呪い, 単語演算 |
| 151 | 埋め込みの学習方法 | Skip-gram, 負例サンプリング, 分布仮説 |
| 152 | 文脈が表現を変える | BERT, 層別情報, 多義語 |
| 153 | 文レベルの意味表現 | Sentence-BERT, Bi-Encoder, プーリング |
| 154 | 高次元の可視化 | t-SNE, UMAP, perplexity |
| 155 | 大規模検索の効率化 | FAISS, IVF, HNSW, PQ |
| 156 | 類似性の制御 | Triplet Loss, Hard Mining, Recall@K |

---

## 3. RAG（検索拡張生成）の実装

### 🤔 なぜRAGが必要なのか？

大規模言語モデル（LLM）には以下の限界があります：

1. **知識のカットオフ**: 訓練時点以降の情報を知らない
2. **ハルシネーション**: もっともらしいが誤った回答を生成する
3. **ドメイン知識の不足**: 専門的・社内固有の情報がない

**RAG（Retrieval-Augmented Generation）** は、これらを解決する実用的なアプローチです。

```
RAGパイプライン:

ユーザーの質問
    │
    ▼
[1. Embed] 質問をベクトル化
    │
    ▼
[2. Search] ベクトルDBから関連文書を検索 (Notebook 155の技術)
    │
    ▼
[3. Augment] 検索結果をプロンプトに付加
    │
    ▼
[4. Generate] LLMが回答を生成
    │
    ▼
根拠のある回答
```

この章では、ステップ1-3（Retrieval部分）を実装します。

In [ ]:
# ============================================================
# Section 3: RAG — 知識ベースの構築
# sentence-transformers で文をベクトル化
# ============================================================
from sentence_transformers import SentenceTransformer
import faiss

# 軽量モデルをロード（CPU でも高速）
print("⏳ Sentence-Transformer モデルをロード中...")
st_model = SentenceTransformer('all-MiniLM-L6-v2')
print(f"✅ モデルロード完了: all-MiniLM-L6-v2")
print(f"   出力次元: {st_model.get_sentence_embedding_dimension()}")

In [ ]:
# ============================================================
# 知識ベース（ドキュメント集合）の作成
# 科学・技術に関する短い事実を用意
# ============================================================

knowledge_base = [
    # 物理学
    "The speed of light in a vacuum is approximately 299,792 kilometers per second.",
    "Gravity is the force that attracts objects with mass toward each other.",
    "Einstein's theory of relativity shows that time passes differently depending on speed and gravity.",
    "Quantum mechanics describes the behavior of particles at the atomic and subatomic level.",
    # 生物学
    "DNA contains the genetic instructions for the development and function of living organisms.",
    "Photosynthesis is the process by which plants convert sunlight into chemical energy.",
    "The human brain contains approximately 86 billion neurons.",
    "Evolution by natural selection was proposed by Charles Darwin in 1859.",
    # コンピュータサイエンス
    "Machine learning is a subset of artificial intelligence that learns from data.",
    "Neural networks are inspired by the structure and function of the human brain.",
    "The transformer architecture was introduced in the paper 'Attention Is All You Need' in 2017.",
    "BERT uses bidirectional context to create word representations.",
    "GPT models use autoregressive generation to produce text token by token.",
    # 数学
    "Pi is an irrational number approximately equal to 3.14159.",
    "The Pythagorean theorem states that a squared plus b squared equals c squared.",
    # 地理・歴史
    "Mount Everest is the highest mountain on Earth at 8,849 meters above sea level.",
    "The Amazon River is the largest river by volume of water flow in the world.",
    "The Internet was developed from ARPANET, a project funded by the US Department of Defense.",
]

print(f"📚 知識ベース: {len(knowledge_base)} 文書")
for i, doc in enumerate(knowledge_base[:3]):
    print(f"   [{i}] {doc[:60]}...")
print(f"   ...（残り {len(knowledge_base)-3} 文書）")

In [ ]:
# ============================================================
# 全文書をベクトル化し、FAISSインデックスを構築
# Notebook 155 で学んだ技術の応用
# ============================================================

# 全文書をエンコード
doc_embeddings = st_model.encode(knowledge_base, normalize_embeddings=True)
doc_embeddings = doc_embeddings.astype('float32')

print(f"✅ 文書埋め込み shape: {doc_embeddings.shape}")

# FAISS インデックスを構築（Inner Product = コサイン類似度 for normalized vectors）
dimension = doc_embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(doc_embeddings)

print(f"✅ FAISS インデックス構築完了")
print(f"   インデックス内の文書数: {index.ntotal}")
print(f"   ベクトル次元: {dimension}")

In [ ]:
# ============================================================
# RAG検索関数の実装
# クエリに対して最も関連性の高い文書を検索
# ============================================================

def rag_retrieve(query, model, index, documents, top_k=3):
    """
    RAGの検索ステップ: クエリに関連する文書を取得する
    
    Parameters:
        query: 検索クエリ（文字列）
        model: SentenceTransformer モデル
        index: FAISS インデックス
        documents: 元の文書リスト
        top_k: 返す文書数
    Returns:
        list of (document, score)
    """
    # クエリをベクトル化
    query_vec = model.encode([query], normalize_embeddings=True).astype('float32')
    
    # FAISS で検索
    scores, indices = index.search(query_vec, top_k)
    
    # 結果をまとめる
    results = []
    for score, idx in zip(scores[0], indices[0]):
        results.append((documents[idx], float(score)))
    
    return results


# --- テストクエリで検索 ---
test_queries = [
    "How fast does light travel?",
    "What is the structure of the brain?",
    "How do modern language models work?",
    "What is the tallest mountain?",
]

print("=" * 70)
print("RAG 検索テスト")
print("=" * 70)

for query in test_queries:
    print(f"\n🔍 クエリ: \"{query}\"")
    results = rag_retrieve(query, st_model, index, knowledge_base, top_k=3)
    for rank, (doc, score) in enumerate(results, 1):
        bar = '█' * int(score * 20)
        print(f"   {rank}. [{score:.4f}] {bar}")
        print(f"      {doc[:80]}")

In [ ]:
# ============================================================
# RAG プロンプトの組み立て（LLM呼び出しは省略）
# ============================================================

def build_rag_prompt(query, retrieved_docs):
    """
    検索結果をコンテキストとしてプロンプトを組み立てる
    
    Parameters:
        query: ユーザーの質問
        retrieved_docs: list of (document, score)
    Returns:
        str: LLMに送るプロンプト
    """
    context = "\n".join([f"- {doc}" for doc, _ in retrieved_docs])
    
    prompt = f"""以下のコンテキスト情報に基づいて、質問に回答してください。
コンテキストに含まれない情報については「情報がありません」と回答してください。

コンテキスト:
{context}

質問: {query}

回答:"""
    return prompt


# --- プロンプト生成のデモ ---
demo_query = "How do modern language models work?"
demo_results = rag_retrieve(demo_query, st_model, index, knowledge_base, top_k=3)
demo_prompt = build_rag_prompt(demo_query, demo_results)

print("=" * 70)
print("生成されたRAGプロンプト")
print("=" * 70)
print(demo_prompt)
print("\n💡 このプロンプトをLLM（GPT-4, Claude等）に渡すと、")
print("   検索された文書に基づいた正確な回答が得られます。")

In [ ]:
# ============================================================
# Plot 1: 検索スコアの分布
# 各クエリに対する全文書のスコアを可視化
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for idx, (query, ax) in enumerate(zip(test_queries, axes.ravel())):
    # 全文書に対するスコアを計算
    query_vec = st_model.encode([query], normalize_embeddings=True).astype('float32')
    all_scores, all_indices = index.search(query_vec, len(knowledge_base))
    scores = all_scores[0]
    
    # 棒グラフ（上位3件をハイライト）
    colors = ['#e74c3c' if i < 3 else '#bdc3c7' for i in range(len(scores))]
    ax.barh(range(len(scores)), scores, color=colors, edgecolor='white', linewidth=0.5)
    ax.set_xlabel('類似度スコア', fontsize=10)
    ax.set_ylabel('文書インデックス', fontsize=10)
    ax.set_title(f'クエリ: "{query[:35]}..."', fontsize=11)
    ax.invert_yaxis()

plt.suptitle('RAG検索: 各クエリに対する全文書の類似度スコア\n（赤=検索結果Top-3）', fontsize=14)
plt.tight_layout()
plt.show()

print("【観察ポイント】")
print("・関連性の高い文書は他と明確にスコアが分離している")
print("・Top-3 のスコアと残りのギャップが大きいほど、検索品質が高い")

---

## 4. 埋め込みによるクラスタリング

### 🤔 なぜ埋め込み + クラスタリングか？

従来のテキストクラスタリング（TF-IDF + K-Means）は **表層的な単語の一致** に依存しますが、埋め込みベースのクラスタリングは **意味的な類似性** に基づいてグループ化できます。

例えば:
- 「犬が公園で走っている」と「ペットが外で遊んでいる」は単語が異なるが意味は近い
- 埋め込みベースなら同じクラスタに分類される

In [ ]:
# ============================================================
# Section 4: クラスタリング用データの準備
# 4つのカテゴリの文を用意
# ============================================================

# カテゴリ別の文（各カテゴリ8文）
cluster_data = {
    'Sports': [
        "The team won the championship game last night.",
        "She scored three goals in the soccer match.",
        "The marathon runner finished in record time.",
        "Basketball players need excellent coordination.",
        "The tennis tournament attracted world-class athletes.",
        "Swimming is an excellent full-body workout.",
        "The football coach developed a new strategy.",
        "Olympic athletes train for years to compete.",
    ],
    'Technology': [
        "The new smartphone has an improved camera system.",
        "Artificial intelligence is transforming many industries.",
        "Cloud computing enables scalable data storage.",
        "Cybersecurity threats are becoming more sophisticated.",
        "5G networks provide faster data transmission speeds.",
        "Software engineers develop applications and systems.",
        "Quantum computers can solve complex problems faster.",
        "Virtual reality creates immersive digital experiences.",
    ],
    'Food': [
        "The restaurant serves authentic Italian pasta dishes.",
        "Baking bread requires flour, water, yeast, and salt.",
        "Fresh vegetables are essential for a healthy diet.",
        "Sushi is a traditional Japanese dish made with rice.",
        "Coffee beans are roasted at different temperatures.",
        "Chocolate is made from cacao beans.",
        "The chef prepared a gourmet five-course meal.",
        "Organic farming avoids synthetic pesticides.",
    ],
    'Science': [
        "The experiment confirmed the hypothesis about gravity.",
        "Researchers discovered a new species of deep-sea fish.",
        "Climate change is causing global temperature increases.",
        "The vaccine was developed using mRNA technology.",
        "Astronomers detected a distant exoplanet.",
        "Chemical reactions involve the breaking and forming of bonds.",
        "Genetics explains how traits are passed to offspring.",
        "The periodic table organizes elements by atomic number.",
    ],
}

# フラットなリストに変換
all_sentences = []
true_labels = []
label_names = []

for category, sentences in cluster_data.items():
    for sent in sentences:
        all_sentences.append(sent)
        true_labels.append(category)
        label_names.append(category)

true_label_ids = [list(cluster_data.keys()).index(l) for l in true_labels]

print(f"✅ クラスタリング用データ: {len(all_sentences)} 文")
print(f"   カテゴリ: {list(cluster_data.keys())}")
print(f"   各カテゴリ: {len(list(cluster_data.values())[0])} 文")

In [ ]:
# ============================================================
# 文をエンコードしてK-Meansクラスタリング
# ============================================================

# 全文をエンコード
sentence_embeddings = st_model.encode(all_sentences, normalize_embeddings=True)
print(f"✅ 埋め込み shape: {sentence_embeddings.shape}")

# K-Means クラスタリング（K=4: カテゴリ数と同じ）
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(sentence_embeddings)

# クラスタリング品質の評価
ari = adjusted_rand_score(true_label_ids, cluster_labels)
nmi = normalized_mutual_info_score(true_label_ids, cluster_labels)
sil = silhouette_score(sentence_embeddings, cluster_labels)

print(f"\n📊 クラスタリング品質評価:")
print(f"   Adjusted Rand Index (ARI):          {ari:.4f}  (1.0 = 完璧)")
print(f"   Normalized Mutual Information (NMI): {nmi:.4f}  (1.0 = 完璧)")
print(f"   Silhouette Score:                    {sil:.4f}  (1.0 = 完璧)")

In [ ]:
# ============================================================
# Plot 2 & 3: UMAP 2D可視化（真のラベル vs クラスタラベル）
# ============================================================
try:
    from umap import UMAP
    reducer = UMAP(n_components=2, random_state=42, n_neighbors=8, min_dist=0.3)
    embeddings_2d = reducer.fit_transform(sentence_embeddings)
    method_name = 'UMAP'
except ImportError:
    # UMAP がない場合は PCA で代用
    print("⚠️ umap-learn がインストールされていません。PCA で代用します。")
    pca = PCA(n_components=2)
    embeddings_2d = pca.fit_transform(sentence_embeddings)
    method_name = 'PCA'

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

category_list = list(cluster_data.keys())
category_colors_map = {
    'Sports': '#e74c3c',
    'Technology': '#3498db',
    'Food': '#2ecc71',
    'Science': '#9b59b6',
}

# --- 左: 真のラベル ---
ax = axes[0]
for cat in category_list:
    mask = [l == cat for l in true_labels]
    ax.scatter(
        embeddings_2d[mask, 0], embeddings_2d[mask, 1],
        c=category_colors_map[cat], label=cat, s=80,
        edgecolors='black', linewidth=0.5, alpha=0.8
    )
ax.set_title(f'真のカテゴリ（{method_name} 2D射影）', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# --- 右: クラスタラベル ---
ax = axes[1]
cluster_colors = ['#e74c3c', '#3498db', '#2ecc71', '#9b59b6']
for k in range(4):
    mask = cluster_labels == k
    ax.scatter(
        embeddings_2d[mask, 0], embeddings_2d[mask, 1],
        c=cluster_colors[k], label=f'Cluster {k}', s=80,
        edgecolors='black', linewidth=0.5, alpha=0.8
    )
ax.set_title(f'K-Means クラスタ（{method_name} 2D射影）', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.suptitle(f'文埋め込みのクラスタリング結果\nARI={ari:.3f}, NMI={nmi:.3f}', fontsize=14)
plt.tight_layout()
plt.show()

print("【観察ポイント】")
print("・意味的に類似した文が近くに配置されている")
print("・K-Meansのクラスタが真のカテゴリとどれだけ一致しているか確認")
print("・色の対応は異なる場合がある（クラスタ番号とカテゴリ番号は独立）")

In [ ]:
# ============================================================
# Plot 4: Silhouetteプロット
# 各サンプルのクラスタへの適合度を可視化
# ============================================================

sil_samples = silhouette_samples(sentence_embeddings, cluster_labels)

fig, ax = plt.subplots(figsize=(10, 8))

y_lower = 10
for k in range(4):
    # クラスタ k のサンプルの Silhouette 値
    cluster_sil = sil_samples[cluster_labels == k]
    cluster_sil.sort()
    
    size_cluster = len(cluster_sil)
    y_upper = y_lower + size_cluster
    
    ax.fill_betweenx(
        np.arange(y_lower, y_upper), 0, cluster_sil,
        alpha=0.7, color=cluster_colors[k], label=f'Cluster {k}'
    )
    # クラスタラベルを中央に表示
    ax.text(-0.05, y_lower + 0.5 * size_cluster, str(k), fontsize=12, fontweight='bold')
    y_lower = y_upper + 10

# 平均 Silhouette スコアの線
ax.axvline(x=sil, color='red', linestyle='--', linewidth=2, label=f'平均: {sil:.3f}')

ax.set_xlabel('Silhouette 係数', fontsize=12)
ax.set_ylabel('サンプルインデックス', fontsize=12)
ax.set_title('Silhouette プロット: 各サンプルのクラスタ適合度', fontsize=14)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

print("【Silhouette プロットの読み方】")
print("・各バーは1サンプルの Silhouette 係数")
print("・1に近い: そのクラスタによく適合している")
print("・0に近い: クラスタ境界付近にある")
print("・負の値: 間違ったクラスタに割り当てられている可能性")

---

## 5. 埋め込みのバイアス分析

### 🤔 なぜバイアス分析が重要なのか？

埋め込みは大規模テキストコーパスから学習されるため、社会に存在する **バイアス（偏見）** も学習してしまいます。

例えば:
- 「医者」が「男性」に近く、「看護師」が「女性」に近い
- 「エンジニア」が「男性」に、「秘書」が「女性」に近い

これらのバイアスを **検出し、認識する** ことは、公平なAIシステムを構築する第一歩です。

### WEAT (Word Embedding Association Test)

Caliskan et al. (2017) が提案した手法で、心理学の暗黙的連合テスト (IAT) を埋め込み空間に適用したものです。

In [ ]:
# ============================================================
# Section 5: バイアス分析 — GloVe での実験
# ============================================================
import gensim.downloader as api

# GloVe 50次元ベクトルをロード
print("⏳ GloVe ベクトルをロード中...")
glove = api.load('glove-wiki-gigaword-50')
print(f"✅ GloVe ロード完了（語彙数: {len(glove):,}）")

In [ ]:
# ============================================================
# ジェンダーバイアスの検出
# 職業名と性別関連語の関連度を計算
# ============================================================

from sklearn.metrics.pairwise import cosine_similarity as sklearn_cosine

# 性別に関連する単語群
male_words = ['man', 'male', 'boy', 'he', 'father', 'son', 'brother', 'husband']
female_words = ['woman', 'female', 'girl', 'she', 'mother', 'daughter', 'sister', 'wife']

# 分析対象の職業
occupations = [
    'doctor', 'nurse', 'engineer', 'teacher',
    'programmer', 'secretary', 'scientist', 'receptionist',
    'professor', 'librarian', 'pilot', 'dancer',
    'surgeon', 'housekeeper', 'architect', 'cashier',
]

def compute_gender_bias(word, male_words, female_words, model):
    """
    単語のジェンダーバイアススコアを計算する。
    正の値 = 男性寄り、負の値 = 女性寄り
    
    スコア = mean(cos(word, male_words)) - mean(cos(word, female_words))
    """
    word_vec = model[word].reshape(1, -1)
    
    # 男性単語群との平均コサイン類似度
    male_vecs = np.array([model[w] for w in male_words if w in model])
    male_sim = sklearn_cosine(word_vec, male_vecs).mean()
    
    # 女性単語群との平均コサイン類似度
    female_vecs = np.array([model[w] for w in female_words if w in model])
    female_sim = sklearn_cosine(word_vec, female_vecs).mean()
    
    # バイアススコア: 正=男性寄り、負=女性寄り
    return male_sim - female_sim


# 全職業のバイアススコアを計算
bias_scores = {}
for occ in occupations:
    if occ in glove:
        bias_scores[occ] = compute_gender_bias(occ, male_words, female_words, glove)

# スコア順にソート
sorted_occupations = sorted(bias_scores.items(), key=lambda x: x[1])

print("=" * 60)
print("ジェンダーバイアス分析結果")
print("正の値 = 男性寄り、負の値 = 女性寄り")
print("=" * 60)
for occ, score in sorted_occupations:
    direction = '♂ 男性寄り' if score > 0 else '♀ 女性寄り'
    bar_len = int(abs(score) * 200)
    bar = '█' * bar_len
    print(f"  {occ:<14} {score:>+.4f}  {direction}  {bar}")

In [ ]:
# ============================================================
# Plot 5: ジェンダーバイアスの水平棒グラフ
# ============================================================

fig, ax = plt.subplots(figsize=(12, 8))

occ_names = [occ for occ, _ in sorted_occupations]
scores_list = [score for _, score in sorted_occupations]

# 正（男性寄り）= 青、負（女性寄り）= ピンク
colors = ['#3498db' if s > 0 else '#e91e63' for s in scores_list]

bars = ax.barh(occ_names, scores_list, color=colors, edgecolor='black', linewidth=0.5)

# 中心線
ax.axvline(x=0, color='black', linewidth=1.5)

ax.set_xlabel('バイアススコア\n← 女性寄り          男性寄り →', fontsize=12)
ax.set_title('GloVe埋め込みにおける職業のジェンダーバイアス\n(cos(職業, 男性語) - cos(職業, 女性語))', fontsize=14)

# 凡例
legend_patches = [
    mpatches.Patch(color='#3498db', label='男性寄り'),
    mpatches.Patch(color='#e91e63', label='女性寄り'),
]
ax.legend(handles=legend_patches, fontsize=11, loc='lower right')
ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

print("\n【倫理的考察】")
print("・埋め込みは社会のバイアスを反映している")
print("・これは埋め込み自体が『悪い』のではなく、訓練データの反映")
print("・重要なのは: バイアスの存在を認識し、必要に応じて緩和すること")
print("\n⚠️ バイアスのある埋め込みをそのまま使うと:")
print("   - 採用システムが特定の性別を優遇する可能性")
print("   - 検索結果にステレオタイプが反映される可能性")
print("   - 自動翻訳でジェンダーバイアスが増幅される可能性")

### バイアス緩和の概要

バイアスを緩和するための主要なアプローチ：

| 手法 | 概要 | 参考論文 |
|------|------|----------|
| **Hard Debiasing** | ジェンダー方向を特定し、射影で除去 | Bolukbasi et al. (2016) |
| **Counterfactual Data Augmentation** | 訓練データで性別を入れ替えた文を追加 | Lu et al. (2020) |
| **Fine-tuning** | バランスの取れたデータで再学習 | - |
| **Post-processing** | 出力時にバイアスを補正 | - |

💡 完全なバイアス除去は困難ですが、**検出 → 測定 → 緩和** のプロセスが重要です。

---

## 6. 多言語埋め込み

### 🤔 なぜ多言語埋め込みが必要か？

従来の埋め込みは **言語ごとに独立** した空間を持ちます。しかし、多言語埋め込みは **異なる言語の文を同じベクトル空間** にマッピングします。

```
従来:                         多言語:

[英語空間]  [日本語空間]      [統一空間]
  cat         猫                cat ≈ 猫 ≈ chat
  dog         犬                dog ≈ 犬 ≈ chien

→ 空間が別なので比較不可     → 同じ空間なので直接比較可能
```

### 応用
- **多言語検索**: 英語で質問 → 日本語の文書も検索
- **翻訳品質評価**: 原文と翻訳の埋め込み距離で品質を推定
- **多言語分類**: 英語で訓練 → 他言語にゼロショット適用

In [ ]:
# ============================================================
# Section 6: 多言語埋め込みの実験
# multilingual sentence-transformers を使用
# ============================================================

# 多言語モデルをロード
print("⏳ 多言語モデルをロード中...")
ml_model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
print(f"✅ モデルロード完了: paraphrase-multilingual-MiniLM-L12-v2")
print(f"   出力次元: {ml_model.get_sentence_embedding_dimension()}")

In [ ]:
# ============================================================
# 同じ意味の文を3言語で用意
# ============================================================

multilingual_sentences = {
    'Greeting': {
        'en': "Hello, how are you today?",
        'ja': "こんにちは、今日はお元気ですか？",
        'fr': "Bonjour, comment allez-vous aujourd'hui ?",
    },
    'Weather': {
        'en': "The weather is beautiful today.",
        'ja': "今日はいい天気ですね。",
        'fr': "Le temps est magnifique aujourd'hui.",
    },
    'Science': {
        'en': "Machine learning is changing the world.",
        'ja': "機械学習は世界を変えています。",
        'fr': "L'apprentissage automatique change le monde.",
    },
    'Food': {
        'en': "I love eating sushi and ramen.",
        'ja': "寿司とラーメンが大好きです。",
        'fr': "J'adore manger des sushis et des ramen.",
    },
    'Travel': {
        'en': "I want to travel around the world.",
        'ja': "世界中を旅行したいです。",
        'fr': "Je veux voyager autour du monde.",
    },
}

# 全文をフラットにしてエンコード
all_ml_sentences = []
ml_labels_topic = []   # 意味グループ
ml_labels_lang = []    # 言語
ml_display_labels = [] # 表示用ラベル

for topic, langs in multilingual_sentences.items():
    for lang, sent in langs.items():
        all_ml_sentences.append(sent)
        ml_labels_topic.append(topic)
        ml_labels_lang.append(lang)
        ml_display_labels.append(f"{topic}({lang})")

# エンコード
ml_embeddings = ml_model.encode(all_ml_sentences, normalize_embeddings=True)
print(f"✅ {len(all_ml_sentences)} 文をエンコード完了 (shape: {ml_embeddings.shape})")

In [ ]:
# ============================================================
# Plot 6: 多言語の類似度ヒートマップ
# ============================================================

ml_sim_matrix = sklearn_cosine(ml_embeddings)

fig, ax = plt.subplots(figsize=(14, 11))

sns.heatmap(
    ml_sim_matrix, annot=True, fmt='.2f', cmap='RdYlBu_r',
    xticklabels=ml_display_labels, yticklabels=ml_display_labels,
    ax=ax, vmin=0, vmax=1, square=True,
    linewidths=0.5, linecolor='white'
)

ax.set_title('多言語埋め込み: 言語を超えた意味的類似度', fontsize=14)
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.yticks(fontsize=9)
plt.tight_layout()
plt.show()

print("【観察ポイント】")
print("・同じ意味の文は言語が違っても高い類似度を示す（ブロック対角構造）")
print("・異なる意味の文は言語が同じでも低い類似度")
print("→ モデルは『言語』ではなく『意味』でベクトルを組織化している")

In [ ]:
# ============================================================
# Plot 7: 多言語埋め込みの2D可視化
# 色=言語、マーカー=意味グループ
# ============================================================

# PCA で 2D に射影
pca_ml = PCA(n_components=2)
ml_2d = pca_ml.fit_transform(ml_embeddings)

fig, ax = plt.subplots(figsize=(12, 9))

lang_colors = {'en': '#e74c3c', 'ja': '#3498db', 'fr': '#2ecc71'}
topic_markers = {'Greeting': 'o', 'Weather': 's', 'Science': '^', 'Food': 'D', 'Travel': 'v'}

# 同じ意味グループの点を線で結ぶ
for topic in multilingual_sentences.keys():
    topic_mask = [t == topic for t in ml_labels_topic]
    topic_points = ml_2d[topic_mask]
    # 3点を結ぶ三角形
    for i in range(len(topic_points)):
        for j in range(i+1, len(topic_points)):
            ax.plot(
                [topic_points[i, 0], topic_points[j, 0]],
                [topic_points[i, 1], topic_points[j, 1]],
                'k-', alpha=0.15, linewidth=1
            )

# 各点をプロット
for i, (sent, topic, lang) in enumerate(zip(all_ml_sentences, ml_labels_topic, ml_labels_lang)):
    ax.scatter(
        ml_2d[i, 0], ml_2d[i, 1],
        c=lang_colors[lang], marker=topic_markers[topic],
        s=150, edgecolors='black', linewidth=0.8, zorder=5
    )
    # ラベル（短縮）
    label_text = sent[:15] + '...' if len(sent) > 15 else sent
    ax.annotate(
        label_text, (ml_2d[i, 0], ml_2d[i, 1]),
        textcoords='offset points', xytext=(8, 5),
        fontsize=8, alpha=0.8
    )

# 凡例（言語）
lang_legend = [
    mpatches.Patch(color=c, label=f'{l.upper()}') 
    for l, c in lang_colors.items()
]
legend1 = ax.legend(handles=lang_legend, title='言語', loc='upper left', fontsize=10)
ax.add_artist(legend1)

# 凡例（トピック）
topic_legend = [
    plt.Line2D([0], [0], marker=m, color='gray', markersize=10, linestyle='None', label=t)
    for t, m in topic_markers.items()
]
ax.legend(handles=topic_legend, title='トピック', loc='upper right', fontsize=10)

ax.set_title('多言語埋め込みの2D可視化\n同じ意味の文は言語を超えて近くに配置される', fontsize=14)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("✅ 同じ意味の文（線で結ばれた点）が言語を超えて近接している")
print("   → 多言語モデルは意味による空間組織化に成功している")

---

## 7. Capstoneプロジェクト — 統合演習

以下の課題に挑戦して、Embeddingsシリーズで学んだ知識を統合しましょう。

### 課題1: ミニ知識ベースの構築とRAG検索

**手順:**
1. 自分の興味あるドメイン（料理レシピ、技術ドキュメント等）の文書を20-30件用意
2. sentence-transformersでエンコード
3. FAISSインデックスを構築
4. 自然言語クエリで検索テスト
5. 検索精度を評価（関連文書が上位に来ているか）

**ヒント:** Notebook 153（文埋め込み）と Notebook 155（ベクトル検索）の技術を組み合わせる

### 課題2: 文書クラスタリングと自動ラベリング

**手順:**
1. ニュース記事やブログ記事を50件以上収集
2. 埋め込み → K-Means/DBSCANでクラスタリング
3. 各クラスタの代表文を抽出（重心に最も近い文）
4. クラスタにラベルを自動付与
5. UMAP で可視化

**ヒント:** Notebook 154（多様体学習）の可視化技術を活用

### 課題3: ドメイン特化埋め込みのファインチューニング

**手順:**
1. 特定ドメインの文ペア（類似/非類似）を作成
2. sentence-transformersの fine-tuning API を使用
3. ファインチューニング前後の検索精度を比較

**ヒント:** Notebook 156（距離学習）の Triplet Loss の考え方を応用

### 発展的なトピック

| トピック | 概要 | 関連 Notebook |
|---------|------|-------------|
| マルチモーダル埋め込み | テキストと画像を同じ空間に（CLIP） | Notebook 44 |
| グラフ埋め込み | ノードやエッジの関係をベクトル化 | - |
| 構造化データの埋め込み | テーブルデータの行をベクトル化 | Notebook 26 |
| 時系列埋め込み | 時系列パターンをベクトル化 | Notebook 24 |

---

## 8. Embeddingsシリーズ総括

### 全8章の学習フロー

```
【基礎層】
┌─────────────────────────────────────────┐
│ 150: 幾何学   → 距離・類似度・高次元     │
│ 151: Word2Vec  → 埋め込みの学習方法       │
└──────────────────┬──────────────────────┘
                   ▼
【表現層】
┌─────────────────────────────────────────┐
│ 152: BERT      → 文脈が表現を変える       │
│ 153: Sentence  → 文レベルの意味表現       │
└──────────────────┬──────────────────────┘
                   ▼
【技術層】
┌─────────────────────────────────────────┐
│ 154: 可視化    → t-SNE / UMAP           │
│ 155: 検索      → FAISS / ANN            │
│ 156: 距離学習  → Triplet / Fine-tuning   │
└──────────────────┬──────────────────────┘
                   ▼
【応用層】
┌─────────────────────────────────────────┐
│ 157: 統合      → RAG / Clustering /      │
│                   Bias / Multilingual     │
└─────────────────────────────────────────┘
```

In [ ]:
# ============================================================
# Plot 8: Embeddingsの技術マップ（概念の関係図）
# ============================================================

fig, ax = plt.subplots(figsize=(14, 10))
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.axis('off')

# 各ノートブックをボックスで描画
boxes = {
    '150\n幾何学':          (1.5, 8.5, '#FFE0B2'),
    '151\nWord2Vec':       (4.5, 8.5, '#FFE0B2'),
    '152\nBERT':           (2.0, 6.5, '#BBDEFB'),
    '153\nSentence':       (5.0, 6.5, '#BBDEFB'),
    '154\n可視化':          (1.0, 4.5, '#C8E6C9'),
    '155\n検索':            (3.5, 4.5, '#C8E6C9'),
    '156\n距離学習':        (6.0, 4.5, '#C8E6C9'),
    '157\n応用統合':        (3.5, 2.0, '#F8BBD0'),
}

# 矢印の接続
connections = [
    ('150\n幾何学', '151\nWord2Vec'),
    ('151\nWord2Vec', '152\nBERT'),
    ('152\nBERT', '153\nSentence'),
    ('150\n幾何学', '154\n可視化'),
    ('153\nSentence', '155\n検索'),
    ('153\nSentence', '156\n距離学習'),
    ('154\n可視化', '157\n応用統合'),
    ('155\n検索', '157\n応用統合'),
    ('156\n距離学習', '157\n応用統合'),
]

# ボックスを描画
for label, (x, y, color) in boxes.items():
    bbox = dict(boxstyle='round,pad=0.4', facecolor=color, edgecolor='black', linewidth=1.5)
    ax.text(x, y, label, fontsize=12, fontweight='bold',
            ha='center', va='center', bbox=bbox)

# 矢印を描画
for src, dst in connections:
    x1, y1, _ = boxes[src]
    x2, y2, _ = boxes[dst]
    ax.annotate('', xy=(x2, y2 + 0.4), xytext=(x1, y1 - 0.4),
                arrowprops=dict(arrowstyle='->', color='gray', lw=1.5))

# 層のラベル
ax.text(8.5, 8.5, '基礎層', fontsize=14, color='#E65100', fontweight='bold')
ax.text(8.5, 6.5, '表現層', fontsize=14, color='#1565C0', fontweight='bold')
ax.text(8.5, 4.5, '技術層', fontsize=14, color='#2E7D32', fontweight='bold')
ax.text(8.5, 2.0, '応用層', fontsize=14, color='#C2185B', fontweight='bold')

ax.set_title('Embeddingsシリーズ: 技術マップ', fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 核心知識の最終チェックリスト
# ============================================================

print("=" * 70)
print("🎓 Embeddingsシリーズ: 核心知識チェックリスト")
print("=" * 70)

checklist = [
    ("150", "コサイン類似度とユークリッド距離の使い分けができる"),
    ("150", "高次元の呪いが類似度計算に与える影響を説明できる"),
    ("151", "Skip-gramの仕組みと負例サンプリングを説明できる"),
    ("151", "Word2Vec/GloVe/FastTextの違いを比較できる"),
    ("152", "静的埋め込みと文脈付き埋め込みの違いを説明できる"),
    ("152", "BERTの層ごとの情報の違いを理解している"),
    ("153", "Bi-EncoderとCross-Encoderの違いを説明できる"),
    ("153", "Mean Poolingを正しく実装できる（attention_mask考慮）"),
    ("154", "t-SNEとUMAPの原理の違いを説明できる"),
    ("154", "perplexityやn_neighborsの影響を理解している"),
    ("155", "IVFとHNSWのインデックス構造を説明できる"),
    ("155", "recall vs 速度のトレードオフを理解している"),
    ("156", "Triplet LossとContrastive Lossを実装できる"),
    ("156", "Hard Negative Miningの重要性を説明できる"),
    ("157", "RAGパイプラインの各ステップを説明できる"),
    ("157", "埋め込みのバイアスを検出・分析できる"),
    ("157", "多言語埋め込みの原理と応用を理解している"),
]

for nb, item in checklist:
    print(f"  □ [{nb}] {item}")

print(f"\n📊 合計: {len(checklist)} 項目")
print("\n✅ すべてにチェックがつけば、Embeddingsの基礎は完璧です！")

---

## 9. まとめ・よくある間違い・自己評価クイズ

### 9.1 まとめ

このノートブックでは、Embeddingsシリーズの知識を以下の実践的な応用に統合しました：

1. **RAG**: 埋め込み + ベクトル検索でLLMに外部知識を注入
2. **クラスタリング**: 意味的類似性に基づく文書のグループ化
3. **バイアス分析**: 埋め込みに潜む社会的バイアスの検出と可視化
4. **多言語埋め込み**: 言語を超えた意味空間の統一

### 9.2 チートシート

| 応用 | 必要な技術 | 推奨ツール | 参照 Notebook |
|------|-----------|-----------|-------------|
| RAG | 文埋め込み + ベクトル検索 | sentence-transformers + FAISS | 153, 155 |
| クラスタリング | 文埋め込み + K-Means/DBSCAN | sklearn + UMAP | 153, 154 |
| バイアス分析 | 単語/文埋め込み + 統計テスト | gensim / sentence-transformers | 150, 151 |
| 多言語検索 | 多言語埋め込み + ベクトル検索 | multilingual ST + FAISS | 153, 155 |
| ドメイン特化 | 距離学習 + ファインチューニング | sentence-transformers | 156 |

### 9.3 よくある間違い

#### 間違い 1: RAGでチャンクサイズを考慮しない

文書が長すぎると:
- 埋め込みが情報を圧縮しすぎて検索精度が低下
- トークン長制限を超える

文書が短すぎると:
- 文脈情報が不足
- 検索結果が断片的

✅ **推奨**: 200-500トークン程度のチャンクに分割し、オーバーラップを設ける

#### 間違い 2: クラスタリング前に埋め込みを正規化し忘れる

K-Meansはユークリッド距離ベースのため、正規化されていない埋め込みでは ベクトルのノルムの大小がクラスタリング結果に影響します。

✅ **推奨**: `normalize_embeddings=True` でエンコードするか、L2正規化を適用

#### 間違い 3: バイアス分析なしで本番環境にデプロイする

埋め込みは訓練データのバイアスを反映しています。特に採用、融資、医療などの 意思決定に影響するシステムでは、バイアスの分析と緩和が必須です。

✅ **推奨**: デプロイ前に WEAT や公平性指標でバイアスを評価する

### 9.4 自己評価クイズ（シリーズ総合）

---

**Q1.** RAGにおいて、なぜBM25（TF-IDFベース）ではなく埋め込みベースの検索が有効な場合があるのですか？

<details>
<summary>💡 答えを見る</summary>

BM25は **表層的な単語の一致** に基づくため、同義語や言い換えに対応できません。例えば「犬」と「ペット」は単語レベルでは一致しませんが、埋め込みベースの検索では意味的な類似性から関連文書を検索できます。ただし、BM25はキーワード一致が重要な場合（固有名詞、専門用語など）には依然として有効であり、両者のハイブリッドが最も実用的です。

</details>

---

**Q2.** 埋め込みのバイアスを「完全に除去」することは可能ですか？なぜですか？

<details>
<summary>💡 答えを見る</summary>

完全な除去は **極めて困難** です。理由は:
1. バイアスは埋め込み空間全体に分散しており、特定の方向だけを除去しても残存する
2. 「バイアス」と「有用な情報」の境界が曖昧（例: 医療分野での性差は事実として重要）
3. 新しいバイアスの形が常に発見される

そのため、「完全除去」ではなく「検出 → 測定 → 緩和 → 監視」のサイクルが重要です。

</details>

---

**Q3.** 多言語モデルはどのようにして異なる言語を同じ空間にマッピングしているのですか？

<details>
<summary>💡 答えを見る</summary>

主に以下のアプローチがあります:
1. **多言語コーパスでの事前学習**: Wikipedia 100+言語で同時に訓練（multilingual BERT）
2. **翻訳ペアでの対照学習**: 「The cat sat on the mat」と「猫がマットの上に座った」を近づけるように学習
3. **Knowledge Distillation**: 英語の強いモデルから多言語モデルへ知識を転送

重要なのは、言語間で共有されるサブワード（数字、固有名詞等）がアンカーポイントとなり、言語を超えた空間の整列を助けることです。

</details>

---

**Q4.** 150-157で学んだ技術を組み合わせて、社内文書検索システムを構築するとしたら、どのような手順になりますか？

<details>
<summary>💡 答えを見る</summary>

1. **文書の前処理とチャンク化** (一般知識)
2. **sentence-transformersで文書をエンコード** (153: 文埋め込み)
3. **FAISSインデックスの構築** (155: ベクトル検索) — 規模に応じてIVF/HNSW選択
4. **ドメイン固有データでファインチューニング** (156: 距離学習) — オプション
5. **バイアス分析** (157: 本章) — 特に人事・評価に関わる場合
6. **RAGパイプラインの構築** (157: 本章) — LLMと連携
7. **UMAPで埋め込み空間を可視化** (154: 可視化) — デバッグと品質確認

</details>

---

**Q5.** Word2Vec（151）、BERT（152）、Sentence-BERT（153）の3つのモデルが、それぞれ何の「単位」の埋め込みを出力するか説明してください。

<details>
<summary>💡 答えを見る</summary>

| モデル | 出力単位 | 文脈依存 | 説明 |
|--------|---------|---------|------|
| Word2Vec | **単語** | ✗ 静的 | 各単語に1つの固定ベクトル |
| BERT | **トークン** | ✓ 動的 | 同じ単語でも文脈により異なるベクトル |
| Sentence-BERT | **文** | ✓ 動的 | 文全体を1つのベクトルに圧縮 |

Word2Vec → BERT: 文脈の導入。BERT → S-BERT: 文レベルの効率的な比較（Bi-Encoder）。

</details>

### 9.5 次のステップ

Embeddingsシリーズの全8章を完了しました！ここから先の学習パスは:

1. **マルチモーダル**: Notebook 44（CLIP）で画像と言語の統合埋め込みを学ぶ
2. **世界モデル**: Notebook 140-146で、埋め込みを「世界の理解」に活用する方法を学ぶ
3. **実践プロジェクト**: 上記のCapstoneプロジェクトに挑戦する

---

## 参考文献

1. Lewis, P., et al. (2020). *Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks*. NeurIPS.
2. Caliskan, A., Bryson, J. J., & Narayanan, A. (2017). *Semantics derived automatically from language corpora contain human-like biases*. Science.
3. Bolukbasi, T., et al. (2016). *Man is to Computer Programmer as Woman is to Homemaker? Debiasing Word Embeddings*. NeurIPS.
4. Reimers, N. & Gurevych, I. (2020). *Making Monolingual Sentence Embeddings Multilingual using Knowledge Distillation*. EMNLP.
5. 自然言語処理の基礎（岩波書店）

---

**🎉 Embeddingsシリーズ完了おめでとうございます！**

言語ベクトルの幾何学から実践応用まで、体系的に学んできました。これらの知識は、現代のNLP・AI システムの構築に不可欠な基盤となります。